### Step 1 - Load + Quick inspection

In [53]:
# NOTE: Path = safer file paths
# NOTE: pandas = read CSV into a DataFrame

from pathlib import Path
import pandas as pd

input_path = Path("practice_dirty_sales.csv")

# NOTE: always confirm path first (avoids waiting time)
print("Exists :", input_path.exists())
print("Absolute PathL :", input_path.resolve())

# NOTE: read CSV into a DataFrame

df = pd.read_csv(input_path)

# NOTE: first look = shape + columns + a few rows
print("Rows, Cols :", df.shape)
print("Columns :", df.columns)

Exists : True
Absolute PathL : C:\Users\pdinh\Python\Python_Full_Course\Python Practice\Pandas-Python\practice_dirty_sales.csv
Rows, Cols : (10, 12)
Columns : Index(['order_id', 'order_time', 'customer_id', 'city', 'state', 'sku', 'qty',
       'unit_price', 'payment_method', 'discount_code', 'line_total',
       'extra_info'],
      dtype='str')


In [54]:
# NOTE: print first 5 rows
df.head()

,order_id,order_time,customer_id,city,state,sku,qty,unit_price,payment_method,discount_code,line_total,extra_info
0,ORD001,2026-01-05,U01,Sydney,NSW,A1,2,$4.50,CARD,NaN,9.00,"{""device"":""ios"",""app_version"":""1.2""}"
1,ORD001,05/01/2026,U01,SYDNEY,nsw,A1,2,4.5 AUD,card,WELCOME10,9.00,"{""device"":""ios"",""app_version"":""1.2""}"
2,ORD002,2026/01/07,U02,Melbourne,VIC,B2,1,7.00,PAYPAL,NaN,7.00,"{""device"":""android"",""app_version"":""2.0""}"
3,ORD003,NaN,U03,Brisbane,QLD,C3,three,5.25,CARD,NaN,15.75,NaN
4,ORD004,2026-01-10T13:20:00Z,U04,Auckland,UNKNOWN,D4,4,"12,00",UNKNOWN,VIP,48.00,"{""device"":""web""}"


### Explore dataset to check any nulls value, duplicate, etc

In [55]:
# NOTE: check for missing values
df.isna().sum()

order_id          0
order_time        1
customer_id       0
city              0
state             0
sku               0
qty               0
unit_price        0
payment_method    0
discount_code     7
line_total        0
extra_info        1
dtype: int64

In [56]:
# NOTE: check for duplicates
print("Duplicates: ",df.duplicated().sum())


Duplicates:  0


In [57]:
# NOTE: check data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        10 non-null     str    
 1   order_time      9 non-null      str    
 2   customer_id     10 non-null     str    
 3   city            10 non-null     str    
 4   state           10 non-null     str    
 5   sku             10 non-null     str    
 6   qty             10 non-null     str    
 7   unit_price      10 non-null     str    
 8   payment_method  10 non-null     str    
 9   discount_code   3 non-null      str    
 10  line_total      10 non-null     float64
 11  extra_info      9 non-null      str    
dtypes: float64(1), str(11)
memory usage: 1.7 KB


### Transform our dataset

In [58]:
df.head()

,order_id,order_time,customer_id,city,state,sku,qty,unit_price,payment_method,discount_code,line_total,extra_info
0,ORD001,2026-01-05,U01,Sydney,NSW,A1,2,$4.50,CARD,NaN,9.00,"{""device"":""ios"",""app_version"":""1.2""}"
1,ORD001,05/01/2026,U01,SYDNEY,nsw,A1,2,4.5 AUD,card,WELCOME10,9.00,"{""device"":""ios"",""app_version"":""1.2""}"
2,ORD002,2026/01/07,U02,Melbourne,VIC,B2,1,7.00,PAYPAL,NaN,7.00,"{""device"":""android"",""app_version"":""2.0""}"
3,ORD003,NaN,U03,Brisbane,QLD,C3,three,5.25,CARD,NaN,15.75,NaN
4,ORD004,2026-01-10T13:20:00Z,U04,Auckland,UNKNOWN,D4,4,"12,00",UNKNOWN,VIP,48.00,"{""device"":""web""}"


In [59]:
df.columns

Index(['order_id', 'order_time', 'customer_id', 'city', 'state', 'sku', 'qty',
       'unit_price', 'payment_method', 'discount_code', 'line_total',
       'extra_info'],
      dtype='str')

In [60]:
# ---- datetime ----
df["order_time"] = pd.to_datetime(df["order_time"], errors="coerce", utc=True)

# ---- text ----
df["city"] = df["city"].str.strip().str.title().astype("string")
df["state"] = df["state"].str.strip().str.upper().astype("string")
df["sku"] = df["sku"].str.strip().str.upper().astype("string")
df["payment_method"] = df["payment_method"].str.strip().str.title().astype("string")
df["discount_code"] = df["discount_code"].str.strip().str.upper().astype("string")

# ---- qty ----
df["qty"] = (
    df["qty"]
    .replace({"three": 3})
    .pipe(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype("Int64")
)

# ---- price ----
df["unit_price"] = (
    df["unit_price"]
    .str.replace(r"[^\d.,]", "", regex=True)
    .str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

df

,order_id,order_time,customer_id,city,state,sku,qty,unit_price,payment_method,discount_code,line_total,extra_info
0,ORD001,2026-01-05 00:00:00+00:00,U01,Sydney,NSW,A1,2,4.50,Card,<NA>,9.00,"{""device"":""ios"",""app_version"":""1.2""}"
1,ORD001,NaT,U01,Sydney,NSW,A1,2,4.50,Card,WELCOME10,9.00,"{""device"":""ios"",""app_version"":""1.2""}"
2,ORD002,NaT,U02,Melbourne,VIC,B2,1,7.00,Paypal,<NA>,7.00,"{""device"":""android"",""app_version"":""2.0""}"
3,ORD003,NaT,U03,Brisbane,QLD,C3,3,5.25,Card,<NA>,15.75,NaN
4,ORD004,NaT,U04,Auckland,UNKNOWN,D4,4,12.00,Unknown,VIP,48.00,"{""device"":""web""}"
5,ORD005,2026-01-11 00:00:00+00:00,-1,Perth,WA,E5,0,0.00,Card,<NA>,0.00,"{""device"":""ios"",""app_version"":null}"
6,ORD006,NaT,U05,Sydney,NSW,F6,1,NaN,Card,<NA>,1200.99,"{""device"":""web"",""campaign"":""summer""}"
7,ORD007,2026-01-12 00:00:00+00:00,U06,Sydney,NSW,A1,2,4.50,Card,VIP,9.00,"{""device"":""ios""}"
8,ORD008,NaT,U07,Melbourne,VIC,B2,1,7.00,Paypal,<NA>,7.00,"{""device"":""android""}"
9,ORD009,2026-01-14 00:00:00+00:00,U07,Melbourne,VIC,B2,1,7.00,Paypal,<NA>,7.00,"{""device"":""android""}"


In [61]:
import json

def parse_json_safe(x):
    if pd.isna(x) or x == "":
        return {}
    try:
        return json.loads(x)
    except:
        return {}

df["extra_dict"] = df["extra_info"].apply(parse_json_safe)


In [62]:
extra_cols = pd.json_normalize(df["extra_dict"])

df = pd.concat([df, extra_cols], axis=1)
df

,order_id,order_time,customer_id,city,state,sku,qty,unit_price,payment_method,discount_code,line_total,extra_info,extra_dict,device,app_version,campaign
0,ORD001,2026-01-05 00:00:00+00:00,U01,Sydney,NSW,A1,2,4.50,Card,<NA>,9.00,"{""device"":""ios"",""app_version"":""1.2""}","{'device': 'ios', 'app_version': '1.2'}",ios,1.2,NaN
1,ORD001,NaT,U01,Sydney,NSW,A1,2,4.50,Card,WELCOME10,9.00,"{""device"":""ios"",""app_version"":""1.2""}","{'device': 'ios', 'app_version': '1.2'}",ios,1.2,NaN
2,ORD002,NaT,U02,Melbourne,VIC,B2,1,7.00,Paypal,<NA>,7.00,"{""device"":""android"",""app_version"":""2.0""}","{'device': 'android', 'app_version': '2.0'}",android,2.0,NaN
3,ORD003,NaT,U03,Brisbane,QLD,C3,3,5.25,Card,<NA>,15.75,NaN,{},NaN,NaN,NaN
4,ORD004,NaT,U04,Auckland,UNKNOWN,D4,4,12.00,Unknown,VIP,48.00,"{""device"":""web""}",{'device': 'web'},web,NaN,NaN
5,ORD005,2026-01-11 00:00:00+00:00,-1,Perth,WA,E5,0,0.00,Card,<NA>,0.00,"{""device"":""ios"",""app_version"":null}","{'device': 'ios', 'app_version': None}",ios,NaN,NaN
6,ORD006,NaT,U05,Sydney,NSW,F6,1,NaN,Card,<NA>,1200.99,"{""device"":""web"",""campaign"":""summer""}","{'device': 'web', 'campaign': 'summer'}",web,NaN,summer
7,ORD007,2026-01-12 00:00:00+00:00,U06,Sydney,NSW,A1,2,4.50,Card,VIP,9.00,"{""device"":""ios""}",{'device': 'ios'},ios,NaN,NaN
8,ORD008,NaT,U07,Melbourne,VIC,B2,1,7.00,Paypal,<NA>,7.00,"{""device"":""android""}",{'device': 'android'},android,NaN,NaN
9,ORD009,2026-01-14 00:00:00+00:00,U07,Melbourne,VIC,B2,1,7.00,Paypal,<NA>,7.00,"{""device"":""android""}",{'device': 'android'},android,NaN,NaN


In [63]:
df["device"] = df["device"].astype("string")
df["app_version"] = df["app_version"].astype("string")
df["campaign"] = df["campaign"].astype("string")


In [64]:
df = df.drop(columns=["extra_info", "extra_dict"])


In [65]:
df

,order_id,order_time,customer_id,city,state,sku,qty,unit_price,payment_method,discount_code,line_total,device,app_version,campaign
0,ORD001,2026-01-05 00:00:00+00:00,U01,Sydney,NSW,A1,2,4.50,Card,<NA>,9.00,ios,1.2,<NA>
1,ORD001,NaT,U01,Sydney,NSW,A1,2,4.50,Card,WELCOME10,9.00,ios,1.2,<NA>
2,ORD002,NaT,U02,Melbourne,VIC,B2,1,7.00,Paypal,<NA>,7.00,android,2.0,<NA>
3,ORD003,NaT,U03,Brisbane,QLD,C3,3,5.25,Card,<NA>,15.75,<NA>,<NA>,<NA>
4,ORD004,NaT,U04,Auckland,UNKNOWN,D4,4,12.00,Unknown,VIP,48.00,web,<NA>,<NA>
5,ORD005,2026-01-11 00:00:00+00:00,-1,Perth,WA,E5,0,0.00,Card,<NA>,0.00,ios,<NA>,<NA>
6,ORD006,NaT,U05,Sydney,NSW,F6,1,NaN,Card,<NA>,1200.99,web,<NA>,summer
7,ORD007,2026-01-12 00:00:00+00:00,U06,Sydney,NSW,A1,2,4.50,Card,VIP,9.00,ios,<NA>,<NA>
8,ORD008,NaT,U07,Melbourne,VIC,B2,1,7.00,Paypal,<NA>,7.00,android,<NA>,<NA>
9,ORD009,2026-01-14 00:00:00+00:00,U07,Melbourne,VIC,B2,1,7.00,Paypal,<NA>,7.00,android,<NA>,<NA>
